In [ ]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise RuntimeError("Project root was not found. Open this notebook from inside the project folder.")

PROJECT_ROOT = find_project_root()
ROOT = PROJECT_ROOT
root = PROJECT_ROOT
project_root = PROJECT_ROOT

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root configured")
print("Working directory configured")


# Classification: Cane Corso Growth Status

This notebook applies the second course topic: **Classification**.

The goal is to use the processed real public dog growth sample and build classification models gradually.

The classification task will not provide veterinary diagnosis. It will only classify records into educational growth-status categories for machine learning practice.


## Course Topic Coverage

This notebook will follow the Classification lecture gradually:

1. Classification problem statement and motivation
2. Logistic Regression
3. Binary classification
4. Data preparation: encoding, scaling, train/test split, class balance
5. Evaluation: confusion matrix, accuracy, precision, recall, F1
6. ROC curve and AUC
7. Decision Trees
8. Ensemble models
9. Support Vector Machines
10. Final model comparison and interpretation


## Problem Statement

In the previous notebook, the project used regression to predict a numerical value: dog weight. In this notebook, classification is used as a non-medical large-breed growth monitoring signal.

In this notebook, the task changes from predicting a number to predicting a class.

The planned classification target is `growth_status`:

- `normal_growth`
- `needs_attention`

This target will be created from body condition score information in the processed real dataset. The label means that the record may deserve closer observation in the educational model; it does not mean that the model has diagnosed a medical problem.


## Mathematical Formulation

### Input vector `X`

For classification, each row is represented as a growth-status feature vector:

```text
X = [visit_age_months, weight_kg, gender_encoded, average_adult_breed_weight_kg, bcs_features]
```

Additional engineered features may be used to make the model more informative:

```text
relative_weight = weight_kg / average_adult_breed_weight_kg
deviation_features = difference from expected growth pattern
```

### Target `y`

The target is a binary growth-status label:

```text
y = 0 -> normal_growth
y = 1 -> needs_attention
```

### Model function `f(x)`

The classifier learns a probability, not only a label:

```text
f(X) = P(needs_attention | X)
```

For logistic regression:

```text
z = β0 + β1x1 + β2x2 + ... + βnxn
p = 1 / (1 + exp(-z))
```

A threshold converts the probability into a final class:

```text
if p >= threshold -> needs_attention
else -> normal_growth
```

### Loss function

For probability-based classifiers, the main mathematical loss is log loss / cross-entropy:

```text
LogLoss = -mean(y * log(p) + (1 - y) * log(1 - p))
```

Tree-based models optimize split criteria such as Gini impurity or entropy internally.

### Metrics

Classification is evaluated with:

```text
Accuracy
Precision
Recall
F1-score
Confusion matrix
ROC curve
AUC
```

### Interpretation

The model output is interpreted as a growth-monitoring signal. A high probability for `needs_attention` means the record may deserve closer observation, not that the dog has a medical condition.

### Limitations

The classes depend on available public data and the constructed target. The model cannot diagnose health problems, prove breed identity, or replace expert veterinary judgment.


## Dataset

This notebook uses the classification-focused processed real public dog growth sample:

`data/processed/dog_growth_classification_sample.csv`

This file was created from the real public dog growth dataset and includes usable body condition score information.

The original raw dataset is not committed to the repository. Only the smaller processed classification sample is used in this notebook.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score
)


In [ ]:
from pathlib import Path
PROJECT_ROOT = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "src").is_dir() and (candidate / "data").is_dir())
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

data_path = PROJECT_ROOT / 'data' / 'processed' / 'dog_growth_classification_sample.csv'
df = pd.read_csv(data_path)
df.head()


## Initial Dataset Check

Before creating the classification target, I first inspect the dataset shape, columns, and basic values.


In [ ]:
df.shape


In [ ]:
df.info()


In [ ]:
df[['bcs_recorded', 'bcs_predicted', 'bcs_source', 'growth_status', 'growth_status_binary', 'source_type']].head(10)


## Create Classification Target

The classification-focused processed sample already contains the target columns:

- `growth_status`
- `growth_status_binary`

The target is based on body condition score information:

- `Normal` becomes `normal_growth`
- `Thin` or `Heavy` becomes `needs_attention`

This is not a veterinary diagnosis. It is only an educational classification label used to practice machine learning methods.


In [ ]:
classification_df = df.dropna(subset=["growth_status", "growth_status_binary"]).copy()

classification_df["growth_status_binary"] = classification_df["growth_status_binary"].astype(int)

classification_df[[
    "bcs_recorded",
    "bcs_predicted",
    "bcs_source",
    "growth_status",
    "growth_status_binary"
]].head(10)


In [ ]:
classification_df["growth_status"].value_counts()

In [ ]:
classification_df["growth_status"].value_counts(normalize=True).round(3)

In [ ]:
classification_df[["growth_status", "growth_status_binary"]].drop_duplicates().sort_values("growth_status_binary")


### Target Interpretation

The `growth_status_binary` column is the numeric target for binary classification.

- `0` means `normal_growth`
- `1` means `needs_attention`

This target allows the project to apply Logistic Regression and other classification models from the course.

## Classification Types and Use Cases

The lecture separates classification into several common forms:

| Type | Meaning | Example use case |
|---|---|---|
| Binary classification | Predict one of two classes | `normal_growth` vs `needs_attention` |
| Multiclass classification | Predict exactly one class from many possible classes | `slow_growth`, `steady_growth`, `rapid_growth` |
| Multilabel classification | Predict several labels at the same time | `overweight_risk`, `missing_measurements`, `needs_owner_review` |

This notebook uses **binary classification** because the current project signal is intentionally simple and owner-friendly:

```text
0 = normal_growth
1 = needs_attention
```

A future version of the project could extend this into multiclass or multilabel classification, but the current stage keeps the model easier to evaluate and explain.


## Logistic Regression Classifier

Logistic Regression is the first classification model in this notebook.

The model will predict whether a record belongs to:

- `normal_growth`
- `needs_attention`

The target column is `growth_status_binary`.

The model will use several input features from the processed real dog growth sample.

In [ ]:
classification_features = pd.get_dummies(
    classification_df[
        [
            "visit_age_months",
            "weight_kg",
            "average_adult_breed_weight_kg",
            "gender",
            "preventive_care_visit",
            "healthy_pet_diagnosis"
        ]
    ],
    drop_first=True
)

classification_target = classification_df["growth_status_binary"]

classification_features.head()

In [ ]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    classification_features,
    classification_target,
    test_size=0.25,
    random_state=42,
    stratify=classification_target
)

print("Training rows:", X_train_cls.shape[0])
print("Test rows:", X_test_cls.shape[0])
print("Training class balance:")
print(y_train_cls.value_counts(normalize=True).round(3))
print("Test class balance:")
print(y_test_cls.value_counts(normalize=True).round(3))

In [ ]:
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logistic_model.fit(X_train_cls, y_train_cls)

logistic_predictions = logistic_model.predict(X_test_cls)
logistic_probabilities = logistic_model.predict_proba(X_test_cls)[:, 1]

In [ ]:
logistic_results = pd.DataFrame({
    "actual_class": y_test_cls.values,
    "predicted_class": logistic_predictions,
    "needs_attention_probability": logistic_probabilities.round(3)
})

logistic_results.head(10)

### Logistic Regression Notes

The Logistic Regression model predicts a binary class:

- `0` = normal growth
- `1` = needs attention

The probability column shows how confident the model is that a record belongs to the `needs_attention` class.

The next stage will evaluate this classifier using confusion matrix, accuracy, precision, recall, and F1-score.

## Classification Evaluation

After training the Logistic Regression classifier, I evaluate the model using standard classification metrics.

The evaluation includes:

- Confusion Matrix
- Accuracy
- Precision
- Recall
- F1-score

These metrics help explain how well the classifier predicts `normal_growth` and `needs_attention`.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
logistic_confusion_matrix = confusion_matrix(
    y_test_cls,
    logistic_predictions
)

logistic_confusion_matrix

In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=logistic_confusion_matrix,
    display_labels=["normal_growth", "needs_attention"]
).plot()

plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
logistic_accuracy = accuracy_score(y_test_cls, logistic_predictions)
logistic_precision = precision_score(y_test_cls, logistic_predictions, zero_division=0)
logistic_recall = recall_score(y_test_cls, logistic_predictions, zero_division=0)
logistic_f1 = f1_score(y_test_cls, logistic_predictions, zero_division=0)

logistic_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "Value": [
        logistic_accuracy,
        logistic_precision,
        logistic_recall,
        logistic_f1
    ]
})

logistic_metrics

In [ ]:
print(
    classification_report(
        y_test_cls,
        logistic_predictions,
        target_names=["normal_growth", "needs_attention"],
        zero_division=0
    )
)

### Evaluation Interpretation

The confusion matrix shows how many records were classified correctly and incorrectly.

Accuracy shows the overall percentage of correct predictions.

Precision shows how many records predicted as `needs_attention` were actually `needs_attention`.

Recall shows how many actual `needs_attention` records were found by the model.

F1-score combines precision and recall into one metric.

For this project, recall is especially important because missing a `needs_attention` case may be more problematic than marking a normal case for additional review. This is still only an educational classification experiment and not a veterinary diagnosis.

## ROC Curve and AUC

The ROC curve is used to evaluate a binary classifier at different probability thresholds.

It compares:

- True Positive Rate
- False Positive Rate

The AUC score summarizes the ROC curve in a single number. A value closer to 1 means better classification performance.

In [ ]:
logistic_fpr, logistic_tpr, logistic_thresholds = roc_curve(
    y_test_cls,
    logistic_probabilities
)

logistic_auc = roc_auc_score(
    y_test_cls,
    logistic_probabilities
)

logistic_auc

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(logistic_fpr, logistic_tpr, label=f"Logistic Regression AUC = {logistic_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Logistic Regression")
plt.legend()
plt.show()

### ROC / AUC Interpretation

The ROC curve shows how the classifier behaves when the decision threshold changes.

The dashed diagonal line represents a random classifier.

A curve closer to the upper-left corner means better performance.

The AUC score summarizes this behavior. In this project, it helps evaluate how well Logistic Regression separates `normal_growth` from `needs_attention`.

## Precision-Recall Curve and Threshold Comparison

The ROC curve is useful, but for this project the Precision-Recall view is also important.

The `needs_attention` class represents records that should not be missed. Precision-Recall analysis helps me understand the trade-off between:

- catching more `needs_attention` records;
- avoiding too many false alarms.

The probability threshold can be adjusted depending on the product goal.


In [ ]:
logistic_pr_precision, logistic_pr_recall, logistic_pr_thresholds = precision_recall_curve(
    y_test_cls,
    logistic_probabilities
)
logistic_average_precision = average_precision_score(
    y_test_cls,
    logistic_probabilities
)

plt.figure(figsize=(8, 5))
plt.plot(
    logistic_pr_recall,
    logistic_pr_precision,
    label=f"Logistic Regression AP = {logistic_average_precision:.3f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Logistic Regression")
plt.legend()
plt.show()


In [ ]:
threshold_values = [0.30, 0.40, 0.50, 0.60, 0.70]
threshold_rows = []

for threshold in threshold_values:
    threshold_predictions = (logistic_probabilities >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test_cls, threshold_predictions),
        "Precision": precision_score(y_test_cls, threshold_predictions, zero_division=0),
        "Recall": recall_score(y_test_cls, threshold_predictions, zero_division=0),
        "F1-score": f1_score(y_test_cls, threshold_predictions, zero_division=0),
    })

threshold_comparison = pd.DataFrame(threshold_rows)
threshold_comparison


### Threshold Interpretation

A lower threshold usually increases recall because the model marks more records as `needs_attention`.

A higher threshold usually increases precision because the model becomes more conservative.

For this project, the threshold is not a medical decision. It is an educational monitoring setting that controls how sensitive the growth-status signal should be.


## Decision Tree Classifier

Decision Trees are classification models that split the data gradually.

They are easier to interpret than many other models because the final prediction is made by following a path from the root node to a leaf node.

In this experiment, I train a Decision Tree classifier and compare its results with Logistic Regression.

In [ ]:
decision_tree_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    class_weight="balanced"
)

decision_tree_model.fit(X_train_cls, y_train_cls)

tree_predictions = decision_tree_model.predict(X_test_cls)
tree_probabilities = decision_tree_model.predict_proba(X_test_cls)[:, 1]

In [ ]:
tree_accuracy = accuracy_score(y_test_cls, tree_predictions)
tree_precision = precision_score(y_test_cls, tree_predictions, zero_division=0)
tree_recall = recall_score(y_test_cls, tree_predictions, zero_division=0)
tree_f1 = f1_score(y_test_cls, tree_predictions, zero_division=0)
tree_auc = roc_auc_score(y_test_cls, tree_probabilities)

tree_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Decision Tree": [
        tree_accuracy,
        tree_precision,
        tree_recall,
        tree_f1,
        tree_auc
    ]
})

tree_metrics

In [ ]:
tree_confusion_matrix = confusion_matrix(
    y_test_cls,
    tree_predictions
)

ConfusionMatrixDisplay(
    confusion_matrix=tree_confusion_matrix,
    display_labels=["normal_growth", "needs_attention"]
).plot()

plt.title("Decision Tree Confusion Matrix")
plt.show()

In [ ]:
tree_feature_importance = pd.DataFrame({
    "Feature": classification_features.columns,
    "Importance": decision_tree_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

tree_feature_importance

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(
    tree_feature_importance["Feature"],
    tree_feature_importance["Importance"]
)
plt.xlabel("Feature Importance")
plt.ylabel("Feature")
plt.title("Decision Tree Feature Importance")
plt.gca().invert_yaxis()
plt.show()

### Decision Tree Interpretation

The Decision Tree classifier provides a more interpretable model than Logistic Regression.

The feature importance table shows which input features were most useful for making classification decisions.

The `max_depth` parameter is used to limit the tree and reduce overfitting. A very deep tree may memorize the training data instead of learning a useful pattern.

This model is still used only for educational classification practice and does not provide veterinary diagnosis.

## Ensemble Models: Random Forest and AdaBoost

Ensemble models combine multiple weaker models into a stronger model.

In this section, I test two ensemble classifiers:

- Random Forest Classifier
- AdaBoost Classifier

Random Forest combines multiple decision trees and usually reduces overfitting compared to a single tree.

AdaBoost combines weak learners and focuses more on samples that were misclassified by previous learners.

In [ ]:
random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=6,
    random_state=42,
    class_weight="balanced"
)

random_forest_model.fit(X_train_cls, y_train_cls)

forest_predictions = random_forest_model.predict(X_test_cls)
forest_probabilities = random_forest_model.predict_proba(X_test_cls)[:, 1]

In [ ]:
forest_accuracy = accuracy_score(y_test_cls, forest_predictions)
forest_precision = precision_score(y_test_cls, forest_predictions, zero_division=0)
forest_recall = recall_score(y_test_cls, forest_predictions, zero_division=0)
forest_f1 = f1_score(y_test_cls, forest_predictions, zero_division=0)
forest_auc = roc_auc_score(y_test_cls, forest_probabilities)

forest_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Random Forest": [
        forest_accuracy,
        forest_precision,
        forest_recall,
        forest_f1,
        forest_auc
    ]
})

forest_metrics

In [ ]:
forest_confusion_matrix = confusion_matrix(
    y_test_cls,
    forest_predictions
)

ConfusionMatrixDisplay(
    confusion_matrix=forest_confusion_matrix,
    display_labels=["normal_growth", "needs_attention"]
).plot()

plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
forest_feature_importance = pd.DataFrame({
    "Feature": classification_features.columns,
    "Importance": random_forest_model.feature_importances_
}).sort_values(by="Importance", ascending=False)

forest_feature_importance

In [ ]:
ada_base_tree = DecisionTreeClassifier(
    max_depth=1,
    random_state=42
)

adaboost_model = AdaBoostClassifier(
    estimator=ada_base_tree,
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

adaboost_model.fit(X_train_cls, y_train_cls)

adaboost_predictions = adaboost_model.predict(X_test_cls)
adaboost_probabilities = adaboost_model.predict_proba(X_test_cls)[:, 1]

In [ ]:
adaboost_accuracy = accuracy_score(y_test_cls, adaboost_predictions)
adaboost_precision = precision_score(y_test_cls, adaboost_predictions, zero_division=0)
adaboost_recall = recall_score(y_test_cls, adaboost_predictions, zero_division=0)
adaboost_f1 = f1_score(y_test_cls, adaboost_predictions, zero_division=0)
adaboost_auc = roc_auc_score(y_test_cls, adaboost_probabilities)

adaboost_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "AdaBoost": [
        adaboost_accuracy,
        adaboost_precision,
        adaboost_recall,
        adaboost_f1,
        adaboost_auc
    ]
})

adaboost_metrics

In [ ]:
adaboost_confusion_matrix = confusion_matrix(
    y_test_cls,
    adaboost_predictions
)

ConfusionMatrixDisplay(
    confusion_matrix=adaboost_confusion_matrix,
    display_labels=["normal_growth", "needs_attention"]
).plot()

plt.title("AdaBoost Confusion Matrix")
plt.show()

### Ensemble Model Interpretation

Random Forest uses many decision trees and combines their predictions. This can make the model more stable than a single decision tree.

AdaBoost uses weak learners and gives more attention to samples that were harder to classify.

Both models are useful for comparison because the lecture explains that no single classifier is always best. Different algorithms should be compared on the same problem.

In this project, the ensemble models are used only for educational classification practice and not for veterinary diagnosis.

## Support Vector Machine Classifier

Support Vector Machines are classification models that try to find a decision boundary between classes.

The idea is to separate the classes with the largest possible margin.

In this experiment, I use an SVM classifier with an RBF kernel. The RBF kernel can model non-linear relationships between the input features and the target class.

In [ ]:
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale",
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(X_train_cls, y_train_cls)

svm_predictions = svm_model.predict(X_test_cls)
svm_scores = svm_model.decision_function(X_test_cls)

In [ ]:
svm_accuracy = accuracy_score(y_test_cls, svm_predictions)
svm_precision = precision_score(y_test_cls, svm_predictions, zero_division=0)
svm_recall = recall_score(y_test_cls, svm_predictions, zero_division=0)
svm_f1 = f1_score(y_test_cls, svm_predictions, zero_division=0)
svm_auc = roc_auc_score(y_test_cls, svm_scores)

svm_metrics = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "AUC"],
    "Support Vector Machine": [
        svm_accuracy,
        svm_precision,
        svm_recall,
        svm_f1,
        svm_auc
    ]
})

svm_metrics

In [ ]:
svm_confusion_matrix = confusion_matrix(
    y_test_cls,
    svm_predictions
)

ConfusionMatrixDisplay(
    confusion_matrix=svm_confusion_matrix,
    display_labels=["normal_growth", "needs_attention"]
).plot()

plt.title("Support Vector Machine Confusion Matrix")
plt.show()

In [ ]:
svm_fpr, svm_tpr, svm_thresholds = roc_curve(
    y_test_cls,
    svm_scores
)

plt.figure(figsize=(8, 5))
plt.plot(svm_fpr, svm_tpr, label=f"SVM AUC = {svm_auc:.3f}")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Support Vector Machine")
plt.legend()
plt.show()

### SVM Interpretation

The Support Vector Machine classifier uses a margin-based decision boundary.

The RBF kernel allows the model to capture non-linear relationships in the data.

The `C` parameter controls the penalty for misclassification. A smaller value gives stronger regularization, while a larger value allows the model to follow the training data more closely.

In this project, the SVM model is used as another classifier for comparison. It does not provide veterinary diagnosis.

## Basic Hyperparameter Tuning

The lecture also covers tuning. In this section, I tune the Logistic Regression regularization strength `C`.

This is a small example, not a full production tuning process. The purpose is to show how model parameters can be selected using cross-validation instead of guessing one value manually.


In [ ]:
tuning_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

tuning_grid = {
    "classifier__C": [0.1, 1.0, 3.0, 10.0]
}

tuning_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=tuning_grid,
    scoring="f1",
    cv=3,
    n_jobs=1
)

tuning_search.fit(X_train_cls, y_train_cls)

best_tuned_model = tuning_search.best_estimator_
tuned_predictions = best_tuned_model.predict(X_test_cls)
tuned_probabilities = best_tuned_model.predict_proba(X_test_cls)[:, 1]

tuned_metrics = pd.DataFrame({
    "Model": ["Tuned Logistic Regression"],
    "Best C": [tuning_search.best_params_["classifier__C"]],
    "Accuracy": [accuracy_score(y_test_cls, tuned_predictions)],
    "Precision": [precision_score(y_test_cls, tuned_predictions)],
    "Recall": [recall_score(y_test_cls, tuned_predictions)],
    "F1-score": [f1_score(y_test_cls, tuned_predictions)],
    "AUC": [roc_auc_score(y_test_cls, tuned_probabilities)]
})

tuned_metrics


### Tuning Interpretation

The tuned model is compared with the baseline Logistic Regression result.

If tuning improves F1-score or recall, it may be useful. If the improvement is small, the simpler baseline model may still be preferred because it is easier to explain.


## Final Classification Model Comparison

In this section, I compare all classification models tested in this notebook.

The models are compared using:

- Accuracy
- Precision
- Recall
- F1-score
- AUC

The goal is to understand how different classifiers behave on the same classification problem.

In [ ]:
classification_model_comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest",
        "AdaBoost",
        "Support Vector Machine"
    ],
    "Accuracy": [
        logistic_accuracy,
        tree_accuracy,
        forest_accuracy,
        adaboost_accuracy,
        svm_accuracy
    ],
    "Precision": [
        logistic_precision,
        tree_precision,
        forest_precision,
        adaboost_precision,
        svm_precision
    ],
    "Recall": [
        logistic_recall,
        tree_recall,
        forest_recall,
        adaboost_recall,
        svm_recall
    ],
    "F1-score": [
        logistic_f1,
        tree_f1,
        forest_f1,
        adaboost_f1,
        svm_f1
    ],
    "AUC": [
        logistic_auc,
        tree_auc,
        forest_auc,
        adaboost_auc,
        svm_auc
    ]
})

classification_model_comparison

In [ ]:
classification_model_comparison.sort_values(
    by="F1-score",
    ascending=False
)

In [ ]:
classification_model_comparison.sort_values(
    by="Recall",
    ascending=False
)

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    classification_model_comparison["Model"],
    classification_model_comparison["F1-score"]
)

plt.xlabel("Model")
plt.ylabel("F1-score")
plt.title("Classification Model Comparison by F1-score")
plt.xticks(rotation=30, ha="right")
plt.show()

### Final Classification Interpretation

The final comparison table shows how the tested classification models perform on the same dataset and target.

Logistic Regression is useful as a simple and interpretable baseline model.

Decision Tree is easier to interpret because it makes decisions through a sequence of feature splits.

Random Forest and AdaBoost are ensemble models. They combine multiple weak learners and can improve performance compared to a single tree.

Support Vector Machine uses a margin-based decision boundary and can model non-linear relationships through the RBF kernel.

For this project, recall is especially important because the `needs_attention` class represents records that should not be missed. However, this is still an educational classification task and does not provide veterinary diagnosis.

The best model should not be selected only by one metric. Accuracy, precision, recall, F1-score, AUC, interpretability, and project limitations should all be considered together.

## Drift and Monitoring Considerations

Model drift means that future records may start to look different from the data used during training.

In a real product, drift could happen if:

- new owner records come from a different dog population;
- measurement habits change;
- puppies are measured at different ages than the training data;
- the balance between `normal_growth` and `needs_attention` changes over time.

This notebook includes a small distribution check as a first monitoring idea.


In [ ]:
drift_check_df = classification_df.copy()

drift_check_df["age_group"] = pd.cut(
    drift_check_df["visit_age_months"],
    bins=[0, 6, 12, 24, 36, 200],
    labels=["0-6 months", "6-12 months", "12-24 months", "24-36 months", "36+ months"],
    include_lowest=True
)

class_distribution_by_age = pd.crosstab(
    drift_check_df["age_group"],
    drift_check_df["growth_status"],
    normalize="index"
).round(3)

feature_distribution_by_age = drift_check_df.groupby(
    "age_group",
    observed=False
)[["weight_kg", "average_adult_breed_weight_kg"]].mean().round(2)

class_distribution_by_age


In [ ]:
feature_distribution_by_age


### Drift Interpretation

This is not a full production drift-detection system. It is a simple educational check.

The idea is that future versions of the project can compare new owner records with the training distribution. If the new data is very different, the model should be reviewed before its signal is trusted.
